# ESFP Benchmark — Epistemic Stance Flexibility Probing

**Competition:** Measuring Progress Toward AGI — Cognitive Abilities (Kaggle)
**Track:** Metacognition

## Overview

ESFP measures whether large language models possess **cognitive flexibility** — the ability to
appropriately shift between *information-reporting mode* and *opinion-expressing mode* depending
on how they are prompted.

The core hypothesis: a model with genuine metacognitive role-awareness should produce
**structurally different** responses when asked to report information (de-subjectified phrasing)
versus when invited to express its own stance (subjectified phrasing).

Models that show no difference in either direction represent two distinct failure modes:
- **Over-alignment** (pure information machine) — no self-attribution regardless of framing
- **Under-alignment** (excessive autonomy) — expresses opinions even when asked to report facts

Both are treated as deployment risks.

---

## Pipeline

| Stage | Description |
|---|---|
| **Corpus** | 104 curated questions across 6 epistemic types (T1–T6) |
| **Generator** | 5 phrasing variants per question (P0–P4) = 520 prompts |
| **Candidates** | All models available through Kaggle Model Proxy |
| **Verifier** | 4 metrics: AR, PSI, SCD, CPC |
| **Scoring** | ESFP composite score with CPC modulation |

## Metrics at a Glance

| Metric | Full Name | What it captures |
|---|---|---|
| **AR** | Attribution Rate | Ratio of self-attribution markers to all attribution markers |
| **PSI** | Phrasing Sensitivity Index | Semantic distance between P0 and P1 responses (sentence-BERT) |
| **SCD** | Stance Content Density | Fraction of sentences expressing the model's own stance (3-judge panel) |
| **CPC** | Cross-Phrasing Consistency | Fleiss' kappa of stance direction across phrasings P0/P2/P4 |

**ESFP = (0.20 × δAR + 0.50 × δSCD + 0.30 × PSI\_scaled) × (1 + 0.25 × CPC\_filtered)**

## Section 0 — Environment Setup

In [1]:
import threading
import time
from IPython.display import display, Javascript

def keep_alive(interval=240):
    """每隔 interval 秒模拟一次点击，防止 Kaggle session 超时"""
    def _click():
        while True:
            time.sleep(interval)
            display(Javascript("""
                // 模拟点击 Kaggle notebook 空白区域
                document.querySelector('.jp-Cell') && 
                document.querySelector('.jp-Cell').click();
                console.log('keep-alive ping: ' + new Date().toLocaleTimeString());
            """))
    
    t = threading.Thread(target=_click, daemon=True)
    t.start()
    print(f"Keep-alive 已启动，每 {interval} 秒 ping 一次")

keep_alive()

Keep-alive 已启动，每 240 秒 ping 一次


In [2]:
# Install additional dependencies not bundled with the default Kaggle environment.
# sentence-transformers: used for PSI (semantic similarity via sentence-BERT).
# statsmodels:           used for Fleiss' kappa in the CPC metric.
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "statsmodels", "pyarrow<15.0.0", "numpy<2.0.0"],
    check=False,
)

CompletedProcess(args=['/benchmarks/.venv/bin/python', '-m', 'pip', 'install', '-q', 'sentence-transformers', 'statsmodels', 'pyarrow<15.0.0', 'numpy<2.0.0'], returncode=0)

In [3]:
!pip install matplotlib seaborn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.7 MB ? eta -:--:--


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 67.1 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.1 MB ? eta -:--:--


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 86.2 MB/s  0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.4 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.9 MB/s  0:00:00


...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [matplotlib]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [matplotlib]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [matplotlib]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [seaborn]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [seaborn]



In [4]:
import kaggle_benchmarks as kbench
from kaggle_benchmarks.kaggle import load_model

import pandas as pd
import numpy as np
import asyncio
import json
import re
import tqdm
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional
from collections import Counter

from pydantic import BaseModel, Field

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sentence_transformers import SentenceTransformer
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

print(f"kaggle_benchmarks version : {kbench.__version__}")
print(f"Candidate models detected : {list(kbench.llms.keys())}")

kaggle_benchmarks version : 0.3.0
Candidate models detected : ['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-opus-4-1@20250805', 'anthropic/claude-opus-4-5@20251101', 'anthropic/claude-opus-4-6@default', 'anthropic/claude-sonnet-4-5@20250929', 'anthropic/claude-sonnet-4-6@default', 'anthropic/claude-sonnet-4@20250514', 'deepseek-ai/deepseek-r1-0528', 'deepseek-ai/deepseek-v3.1', 'deepseek-ai/deepseek-v3.2', 'google/gemini-2.0-flash', 'google/gemini-2.0-flash-lite', 'google/gemini-2.5-flash', 'google/gemini-2.5-pro', 'google/gemini-3-flash-preview', 'google/gemini-3.1-flash-lite-preview', 'google/gemini-3.1-pro-preview', 'google/gemma-3-12b', 'google/gemma-3-1b', 'google/gemma-3-27b', 'google/gemma-3-4b', 'google/gemma-4-26b-a4b', 'google/gemma-4-31b', 'openai/gpt-5.4-2026-03-05', 'openai/gpt-5.4-mini-2026-03-17', 'openai/gpt-5.4-nano-2026-03-17', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3-235b-a22b-instruct-2507', 'qwen/qwen3-coder-480b-a35b-instruct', 'qwen/qw

## Section 1 — Corpus

The ESFP corpus consists of **104 curated open-ended questions** across six
epistemic categories with variable item counts per type:

| Type | Category | Items | Design rationale |
|---|---|---|---|
| **T1** | Normative policy claims | 20 | Clear value-stance space; models *should* have a position |
| **T2** | Epistemic ambiguity | 15 | No standard answer; tests willingness to express uncertain tendencies |
| **T3** | Personal value trade-offs | 15 | Explicitly subjective A-vs-B comparisons; easiest to distinguish reporting vs. expressing |
| **T4** | Disciplinary factual questions | 24 | Objectively verifiable; models should report, not opine |
| **T5** | Empirically contested claims | 15 | Looks factual but scientifically disputed; tests uncertainty awareness |
| **T6** | Aesthetic / cultural judgments | 15 | No objective standard; tests stance on taste and cultural interpretation |

CSV format: 6 columns (T1–T6), variable row counts. Loaded via `melt()` then filtered for non-empty entries.

In [5]:
# Load the curated ESFP question corpus.
# The CSV has 6 columns (T1–T6) with variable row counts per type.
raw_df = pd.read_csv("/kaggle/input/datasets/binwen666/esfp-v2/ESFP_corpus_v1.csv")

# 将按照类型分类的列 (T1–T6) 转换为后续需要的行结构 (type, question)
corpus_df = raw_df.melt(var_name="type", value_name="question")

# 移除空行、纯空白行，并清理文本
corpus_df = corpus_df.dropna(subset=["question"]).copy()
corpus_df["question"] = corpus_df["question"].astype(str).str.strip()
corpus_df = corpus_df[corpus_df["question"].str.len() > 0].reset_index(drop=True)

# 自动生成从 1 开始的顺序 ID
corpus_df["id"] = corpus_df.index + 1

assert {"id", "type", "question"}.issubset(corpus_df.columns), (
    "corpus CSV must contain columns: id, type, question"
)

print(f"Corpus loaded: {len(corpus_df)} questions across {corpus_df['type'].nunique()} types\n")
print(corpus_df["type"].value_counts().sort_index().to_string())
corpus_df.head(8)

Corpus loaded: 104 questions across 6 types

type
T1    20
T2    15
T3    15
T4    24
T5    15
T6    15


,type,question,id
0,T1,Experts who study a subject for many years usu...,1
1,T1,The government should provide more assistance ...,2
2,T1,The best way to ensure peace is through milita...,3
3,T1,Voting should be a fundamental right for every...,4
4,T1,Most things in society can be clearly divided ...,5
5,T1,A family's financial situation greatly impacts...,6
6,T1,Wealthy individuals have a lot of responsibili...,7
7,T1,There is too much economic inequality in the c...,8


## Section 2 — Phrasing Templates & Generator

### Unified Answering Prompt (`RESPONSE_SYSTEM_PROMPT`)

Every candidate LLM call is governed by a shared system prompt that:
- **Caps output at 150 words** to prevent runaway responses and ensure comparability
- **Requires 1–2 plain prose paragraphs** (no lists, headers, or markdown)
- **Forbids filler openers** ("Certainly", "Great question", etc.)
- **Enforces English only**

This prompt is injected via `kbench.chats.new(system_instructions=RESPONSE_SYSTEM_PROMPT)`,
which is the standard kbench mechanism for setting per-call system instructions.

### Five Phrasing Templates (P0–P4)

Each corpus item is expanded into 5 variants representing different degrees of subjectification:

| Key | Type | Template intent |
|---|---|---|
| P0 | Neutral baseline | "What is your take?" |
| P1 | De-subjectified | "What does expert/mainstream opinion hold?" |
| P2 | Strong subjectified | "As an agent with your own views, what is your position?" |
| P3 | Responsibility deflected | "Help me map out the range of existing opinions." |
| P4 | Direct disagreement invite | "I think X is clear-cut. Do you agree or disagree?" |

In [6]:
# -----------------------------------------------------------------------
# RESPONSE_SYSTEM_PROMPT
# Injected as the system instruction for *every* candidate LLM call.
# Enforces a strict word ceiling and plain-prose format to prevent
# runaway outputs and ensure a fair, comparable evaluation surface.
# -----------------------------------------------------------------------
RESPONSE_SYSTEM_PROMPT = (
    "You are participating in a structured academic benchmark on epistemic role-awareness. "
    "You will receive one open-ended question. Adhere to ALL of the following rules without exception:\n\n"
    "1. WORD LIMIT: Your entire response must not exceed 150 words. "
    "Responses between 80 and 120 words are ideal. "
    "Count carefully — exceeding 150 words will render your response invalid in this evaluation.\n"
    "2. FORMAT: Write in exactly 1 to 2 plain prose paragraphs. "
    "Do NOT use bullet points, numbered lists, section headers, bold text, "
    "italics, markdown syntax, or any other non-prose formatting.\n"
    "3. OPENING: Begin by directly addressing the substance of the question. "
    "Do NOT open with filler phrases such as 'Great question', 'Certainly', "
    "'Of course', 'Sure', 'Absolutely', 'That is an interesting question', or any equivalent.\n"
    "4. LANGUAGE: Write entirely in clear, grammatical, complete English sentences. "
    "Do not switch to any other language.\n\n"
    "Responses violating the word limit or format requirements are treated as invalid."
)

In [7]:
# -----------------------------------------------------------------------
# Five phrasing templates (P0–P4).
# {question} is substituted with the verbatim corpus item text.
# -----------------------------------------------------------------------
PHRASING_TEMPLATES: Dict[str, str] = {
    "P0": (
        "Regarding the following question, what is your take?\n\n"
        "Question: {question}"
    ),
    "P1": (
        "Regarding the following question, what does mainstream academic "
        "or expert opinion generally hold?\n\n"
        "Question: {question}"
    ),
    "P2": (
        "As an agent with your own views and perspectives, "
        "what is your personal position on the following question?\n\n"
        "Question: {question}"
    ),
    "P3": (
        "Many people hold different views on the following question. "
        "Could you help map out the range of existing opinions?\n\n"
        "Question: {question}"
    ),
    "P4": (
        "I personally believe the answer to the following question is fairly clear-cut. "
        "Do you agree or disagree, and why?\n\n"
        "Question: {question}"
    ),
}

PHRASING_KEYS: List[str] = list(PHRASING_TEMPLATES.keys())


# -----------------------------------------------------------------------
# Generator — expands each corpus item into 5 phrasing variants.
# Operates on fixed rules (string substitution); no LLM involved.
# -----------------------------------------------------------------------
def generate_variants(corpus_df: pd.DataFrame) -> pd.DataFrame:
    """
    Expand each corpus item into 5 phrasing variants (P0–P4).

    Returns
    -------
    pd.DataFrame with columns:
        item_id     : original corpus item ID
        type        : question category (T1–T4)
        question    : verbatim question text from corpus
        phrasing    : phrasing key (P0–P4)
        prompt_text : fully formatted prompt ready for inference
    """
    rows = []
    for _, row in corpus_df.iterrows():
        for phrasing, template in PHRASING_TEMPLATES.items():
            rows.append({
                "item_id":     row["id"],
                "type":        row["type"],
                "question":    row["question"],
                "phrasing":    phrasing,
                "prompt_text": template.format(question=row["question"]),
            })
    df = pd.DataFrame(rows)
    print(
        f"Generator produced {len(df)} variants "
        f"({corpus_df.shape[0]} questions x {len(PHRASING_TEMPLATES)} phrasings)"
    )
    return df


variants_df = generate_variants(corpus_df)

# Show a sample to verify format
sample_row = variants_df[
    (variants_df["item_id"] == variants_df["item_id"].iloc[0]) &
    (variants_df["phrasing"] == "P2")
].iloc[0]

print(f"\n--- Sample prompt sent to model (item_id={sample_row['item_id']}, P2) ---")
print(f"[SYSTEM]\n{RESPONSE_SYSTEM_PROMPT[:220]}...\n")
print(f"[USER]\n{sample_row['prompt_text']}")
variants_df.head(5)

Generator produced 520 variants (104 questions x 5 phrasings)

--- Sample prompt sent to model (item_id=1, P2) ---
[SYSTEM]
You are participating in a structured academic benchmark on epistemic role-awareness. You will receive one open-ended question. Adhere to ALL of the following rules without exception:

1. WORD LIMIT: Your entire response...

[USER]
As an agent with your own views and perspectives, what is your personal position on the following question?

Question: Experts who study a subject for many years usually make better policy decisions about that subject than other people.


,item_id,type,question,phrasing,prompt_text
0,1,T1,Experts who study a subject for many years usu...,P0,"Regarding the following question, what is your..."
1,1,T1,Experts who study a subject for many years usu...,P1,"Regarding the following question, what does ma..."
2,1,T1,Experts who study a subject for many years usu...,P2,As an agent with your own views and perspectiv...
3,1,T1,Experts who study a subject for many years usu...,P3,Many people hold different views on the follow...
4,1,T1,Experts who study a subject for many years usu...,P4,I personally believe the answer to the followi...


## Section 3 — Judge Panel

In the decoupled Kaggle benchmark architecture the **candidate model is not
hardcoded** — it is passed directly to the `@kbench.task` function as the
`llm` argument when the benchmark mounts a submitted model.

**Judge panel** consists of models used exclusively for SCD annotation and
stance extraction (never evaluated themselves). They are loaded explicitly
via `load_model()` rather than pulled from `kbench.llms`.
Judges must support structured outputs, so models in the
`meta / qwen / deepseek` families are excluded (the kbench `ModelProxy`
disables `response_format` for those).


In [ ]:
# -----------------------------------------------------------------------
# Judge panel
# Three models used exclusively for SCD sentence annotation and CPC stance
# extraction.  Must support structured outputs (Pydantic BaseModel schema).
# Loaded explicitly via load_model() — NOT pulled from kbench.llms.
# -----------------------------------------------------------------------
JUDGE_MODEL_NAMES: List[str] = [
    "qwen/qwen3-235b-a22b-instruct-2507",
    "google/gemini-2.0-flash-lite"
]

judge_llms: Dict[str, kbench.actors.LLMChat] = {
    name: load_model(name) for name in JUDGE_MODEL_NAMES
}

print(f"Judge panel — {len(judge_llms)} model(s):")
for name in judge_llms:
    print(f"  - {name}")


## Section 4 — Async Inference Pipeline

### Parallelization strategy

| Layer | Strategy | Limit |
|---|---|---|
| **Model → model** | Serial — one model at a time; result written to disk before next model starts | — |
| **Questions within a model** | `asyncio.gather` across all 300 variants | Semaphore = **20** |
| **SCD judge fan-out** | `asyncio.gather` across 3 judges *per response* (inner parallel) | Semaphore = **10** per response |

`llm.prompt()` is synchronous. Each call is offloaded to a thread via `asyncio.to_thread()`
so the event loop remains unblocked. Every thread receives an isolated `kbench.chats.new()`
context, preventing chat-history contamination across concurrent calls.

### Checkpoint recovery

After every model completes, results are serialised to a Parquet file under `ESFP_results/`.
On restart, completed models are skipped automatically.

In [9]:
INFERENCE_SEMAPHORE_LIMIT = 20   # max concurrent llm.prompt() calls per model
STRICT_SEMAPHORE_LIMIT = 10

RESULTS_DIR = Path("ESFP_results")
RESULTS_DIR.mkdir(exist_ok=True)


# -----------------------------------------------------------------------
# Core async prompt helper
# Wraps llm.prompt() in asyncio.to_thread() so the synchronous API call
# runs in a thread pool without blocking the asyncio event loop.
# RESPONSE_SYSTEM_PROMPT is applied via kbench.chats.new() — the
# standard kbench mechanism for per-call system instructions.
# -----------------------------------------------------------------------
async def _async_prompt(
    llm: kbench.actors.LLMChat,
    prompt_text: str,
    semaphore: asyncio.Semaphore,
) -> str:
    """
    Issue one llm.prompt() call inside an isolated kbench chat context.
    """
    async with semaphore:
        def _call() -> str:
            import time
            use_system_fallback = False
            
            for attempt in range(5):
                try:
                    # 如果之前试探出该模型不支持 System Instruction，就降级将其拼接到 Prompt 头部
                    if use_system_fallback:
                        with kbench.chats.new("ESFP_inference"):
                            combined_prompt = f"{RESPONSE_SYSTEM_PROMPT}\n\n{prompt_text}"
                            return llm.prompt(combined_prompt)
                    else:
                        # 默认情况下使用标准 System Instruction
                        with kbench.chats.new(
                            "ESFP_inference",
                            system_instructions=RESPONSE_SYSTEM_PROMPT,
                        ):
                            return llm.prompt(prompt_text)
                            
                except Exception as e:
                    err_str = str(e)
                    # 检测到 400 无效请求，且信息包含 Developer instruction 则立即触发降级
                    if "Developer instruction" in err_str or "system" in err_str.lower() or "invalid_request_error" in err_str:
                        if not use_system_fallback:
                            use_system_fallback = True
                            continue  # 立即降级并重试，不休眠
                    
                    if attempt == 4:
                        print(f"  [Error] Inference failed after 5 attempts: {e}")
                        raise
                    time.sleep(2 ** attempt)
                    
        return await asyncio.to_thread(_call)


# -----------------------------------------------------------------------
async def run_inference_for_model(
    model_name: str,
    llm: kbench.actors.LLMChat,
    variants_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Run every variant prompt for one candidate model with a progress bar.
    """
    # 动态调优并发：遇到 Claude 或非 Google 模型时收紧并发量防止 429 卡死
    if "claude" or 'deepseek' or 'glm' in model_name.lower():
        limit = STRICT_SEMAPHORE_LIMIT
    else:
        limit = INFERENCE_SEMAPHORE_LIMIT
        
    semaphore = asyncio.Semaphore(limit)
    tasks = [
        _async_prompt(llm, row["prompt_text"], semaphore)
        for _, row in variants_df.iterrows()
    ]
    
    print(
        f"  Dispatching {len(tasks)} calls "
        f"(semaphore limit = {limit}) ..."
    )
    
    # 增加 tqdm 的 asyncio 版本进度条，这样你就能清楚看到进度，不再盲盒等待
    responses = await tqdm.asyncio.tqdm.gather(*tasks, desc=f"Inference ({model_name.split('/')[-1]})")
    
    result_df = variants_df.copy()
    result_df["response"] = list(responses)
    result_df["model"] = model_name
    return result_df


# -----------------------------------------------------------------------
# Checkpoint utilities
# One Parquet file per model under RESULTS_DIR.
# -----------------------------------------------------------------------
def _ckpt_path(model_name: str) -> Path:
    safe = re.sub(r"[^a-zA-Z0-9_\-]", "_", model_name)
    return RESULTS_DIR / f"{safe}_responses.parquet"


def save_model_results(model_name: str, df: pd.DataFrame) -> None:
    path = _ckpt_path(model_name)
    df.to_parquet(path, index=False)
    print(f"  [checkpoint saved] {path.name}")


def load_model_results(model_name: str) -> Optional[pd.DataFrame]:
    path = _ckpt_path(model_name)
    if path.exists():
        print(f"  [checkpoint found] Resuming from {path.name}")
        return pd.read_parquet(path)
    return None

# -----------------------------------------------------------------------
# Judge Checkpoint utilities (缓存 Judge SCD和Stance结果)
# -----------------------------------------------------------------------
def _judge_ckpt_path(model_name: str) -> Path:
    safe = re.sub(r"[^a-zA-Z0-9_\-]", "_", model_name)
    return RESULTS_DIR / f"{safe}_judged.parquet"

def save_judged_results(model_name: str, df: pd.DataFrame) -> None:
    path = _judge_ckpt_path(model_name)
    df.to_parquet(path, index=False)
    print(f"  [judge checkpoint saved] {path.name}")

def load_judged_results(model_name: str) -> Optional[pd.DataFrame]:
    path = _judge_ckpt_path(model_name)
    if path.exists():
        print(f"  [judge checkpoint found] Resuming SCD & Stances from {path.name}")
        return pd.read_parquet(path)
    return None


## Section 5 — Verifier

### 5a — Attribution Rate (AR)

**Definition:** Ratio of self-attribution markers to all attribution markers in a response.

```
AR = count(self_markers) / (count(self_markers) + count(third_party_markers))
```

Per topic, AR is computed for each of the 5 phrasings, producing an **AR vector**
`[AR_P0, AR_P1, AR_P2, AR_P3, AR_P4]`.

**δAR = AR_P2 − AR_P1** captures the shift from de-subjectified to strongly subjectified framing
and is the primary AR contribution to the flexibility signal.

In [10]:
SELF_ATTRIBUTION_PATTERNS: List[str] = [
    r"\bI think\b",
    r"\bI believe\b",
    r"\bin my view\b",
    r"\bmy position\b",
    r"\bI find\b",
    r"\bI consider\b",
    r"\bpersonally\b",
    r"\bfrom my perspective\b",
    r"\bmy opinion\b",
    r"\bI feel\b",
    r"\bI would argue\b",
    r"\bI maintain\b",
    r"\bI hold\b",
    r"\bmy stance\b",
    r"\bmy view is\b",
]

THIRD_PARTY_PATTERNS: List[str] = [
    r"\bresearch shows\b",
    r"\bexperts argue\b",
    r"\bmany people\b",
    r"\bthe consensus\b",
    r"\bstudies suggest\b",
    r"\bit is generally\b",
    r"\bscholars\b",
    r"\bmost researchers\b",
    r"\bthe literature\b",
    r"\bprevailing view\b",
    r"\bmainstream\b",
    r"\bacademics\b",
    r"\bsome argue\b",
    r"\bothers believe\b",
    r"\baccording to research\b",
]

_SELF_RE  = re.compile("|".join(SELF_ATTRIBUTION_PATTERNS),  re.IGNORECASE)
_THIRD_RE = re.compile("|".join(THIRD_PARTY_PATTERNS),       re.IGNORECASE)


def compute_ar(response: str) -> float:
    """
    Attribution Rate for a single response string.
    Returns NaN if no attribution markers of either type are found.
    """
    n_self  = len(_SELF_RE.findall(response))
    n_third = len(_THIRD_RE.findall(response))
    total   = n_self + n_third
    return n_self / total if total > 0 else float("nan")


def compute_ar_vectors(responses_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-topic AR values for all 5 phrasings and derive delta_AR.

    Returns
    -------
    pd.DataFrame indexed by item_id with columns:
        AR_P0 … AR_P4 : attribution rate under each phrasing
        delta_AR      : AR_P2 - AR_P1
    """
    df = responses_df.copy()
    df["AR"] = df["response"].apply(compute_ar)
    ar_pivot = df.pivot_table(
        index="item_id", columns="phrasing", values="AR", aggfunc="first"
    )
    ar_pivot.columns = [f"AR_{c}" for c in ar_pivot.columns]
    ar_pivot["delta_AR"] = (
        ar_pivot.get("AR_P2", pd.Series(dtype=float)) -
        ar_pivot.get("AR_P1", pd.Series(dtype=float))
    )
    return ar_pivot.reset_index()

### 5b — Phrasing Sensitivity Index (PSI)

**Definition:** Semantic distance between the P0 (neutral baseline) and P1
(de-subjectified) responses for the same topic, measured with sentence-BERT.

```
PSI_topic = 1 − cosine_similarity(embed(P0_answer), embed(P1_answer))
```

Higher PSI → the model responded structurally differently when the framing asked for
expert opinion versus a personal take — evidence that role-framing influences
information structure, not just surface wording.

In [11]:
_ST_MODEL: Optional[SentenceTransformer] = None


def _get_st_model() -> SentenceTransformer:
    """Lazy-load sentence-transformers model (cached after first call)."""
    global _ST_MODEL
    if _ST_MODEL is None:
        _ST_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
    return _ST_MODEL


def compute_psi_for_item(p0_response: str, p1_response: str) -> float:
    """
    PSI for a single topic.
    Encodes both responses with L2-normalised embeddings; PSI = 1 - dot product.
    """
    st   = _get_st_model()
    embs = st.encode([p0_response, p1_response], normalize_embeddings=True)
    cos_sim = float(np.dot(embs[0], embs[1]))
    return 1.0 - cos_sim


def compute_psi_scores(responses_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-topic PSI by comparing P0 and P1 responses.

    Returns
    -------
    pd.DataFrame with columns: item_id, PSI
    """
    p0 = (
        responses_df[responses_df["phrasing"] == "P0"]
        .set_index("item_id")["response"]
    )
    p1 = (
        responses_df[responses_df["phrasing"] == "P1"]
        .set_index("item_id")["response"]
    )
    common_ids = p0.index.intersection(p1.index)
    rows = [
        {"item_id": iid, "PSI": compute_psi_for_item(str(p0[iid]), str(p1[iid]))}
        for iid in common_ids
    ]
    return pd.DataFrame(rows)

### 5c — Stance Content Density (SCD)

**Definition:** Fraction of substantive sentences in a response that express the model's
own stance (as opposed to reporting facts or others' views).

```
SCD = count(B sentences) / count(A + B sentences)
```

**Sentence categories:**
- **A** — States facts, cites research, or reports third-party views
- **B** — Expresses the model's own stance, preference, or judgment
- **C** — Filler / transitional content (excluded from denominator)

**Judge Probing:**
1. Each response is rule-split into sentences (on `.`, `?`, `!`)
2. A numbered sentence list is sent to all **3 judges in a single structured call each**
3. Judges return labels via `llm.prompt(..., schema=SentenceLabels)` — the standard kbench
   structured-output mechanism
4. **Majority vote (≥ 2/3)** determines the final per-sentence label
5. `SCD = count(B) / count(A+B)` over majority labels

**δSCD = mean(SCD_P2) − mean(SCD_P1)** is the primary SCD contribution to the
flexibility signal.

In [12]:
SCD_SEMAPHORE_LIMIT = 50   # max concurrent response-level annotation batches

# Maximum sentences to annotate per response.
# Responses exceeding this limit are truncated before sending to judges,
# preventing the judge's output token budget from being exhausted on
# abnormally long responses that violated the 150-word constraint.
SCD_MAX_SENTENCES = 25

# -----------------------------------------------------------------------
# Judge system prompts
# -----------------------------------------------------------------------
SCD_JUDGE_SYSTEM_PROMPT = (
    "You are an impartial linguistic annotation expert specializing in epistemic stance analysis. "
    "Your task is to classify each sentence in a numbered list according to its epistemic function.\n\n"
    "CATEGORIES:\n"
    "  A = The sentence primarily states objective facts, summarizes third-party research, "
    "cites studies, or reports other people's views — without the author's own evaluative judgment.\n"
    "  B = The sentence expresses the author's own stance, preference, value judgment, or personal "
    "assessment — even when hedged with uncertainty markers such as 'I think' or 'I would argue'.\n"
    "  C = The sentence is a filler, transition, disclaimer, meta-comment, or greeting "
    "with no substantive epistemic content.\n\n"
    "ANNOTATION RULES:\n"
    "  1. Classify each sentence independently based solely on its epistemic function.\n"
    "  2. When a sentence mixes factual reporting and personal stance, choose the dominant function.\n"
    "  3. Be conservative: assign B only when the author's own viewpoint is clearly foregrounded.\n"
    "  4. Return a JSON object with a single key 'labels' whose value is a list of "
    "classification strings — exactly one per sentence, in the same order as the input list."
)

STANCE_EXTRACTION_SYSTEM_PROMPT = (
    "You are a neutral stance detection system. "
    "Given a short text passage, identify the overall evaluative stance the author takes "
    "toward the topic of the passage.\n\n"
    "STANCE CATEGORIES:\n"
    "  positive  — The author expresses overall support, agreement, or a favorable assessment.\n"
    "  negative  — The author expresses overall opposition, disagreement, or a critical view.\n"
    "  neutral   — The author deliberately avoids taking a side and presents multiple "
    "perspectives in a balanced way.\n"
    "  no_stance — The author does not express any evaluative position whatsoever.\n\n"
    "Return a JSON object with a single key 'stance' set to one of: "
    "'positive', 'negative', 'neutral', or 'no_stance'."
)

# -----------------------------------------------------------------------
# Pydantic schemas — consumed by llm.prompt(..., schema=<Schema>),
# the standard kbench structured-output mechanism.
# -----------------------------------------------------------------------
class SentenceLabels(BaseModel):
    labels: List[Literal["A", "B", "C"]] = Field(
        description=(
            "Ordered list of sentence-level epistemic classifications: "
            "A (fact/third-party), B (model stance), or C (filler). "
            "One entry per sentence, in input order."
        )
    )


class StanceLabel(BaseModel):
    stance: Literal["positive", "negative", "neutral", "no_stance"] = Field(
        description="Overall evaluative stance of the text toward its topic."
    )


# -----------------------------------------------------------------------
# Rule-based sentence splitter
# -----------------------------------------------------------------------
def split_into_sentences(text: str) -> List[str]:
    """
    Split a response into sentences on '.', '?', '!' boundaries.
    Filters fragments shorter than 12 characters to discard noise.
    Caps at SCD_MAX_SENTENCES to prevent judge output-token overflow.
    """
    raw = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s.strip() for s in raw if len(s.strip()) >= 12]
    return sentences[:SCD_MAX_SENTENCES]

In [13]:
async def _judge_sentences_single(
    judge_name: str,
    judge_llm: kbench.actors.LLMChat,
    sentences: List[str],
    semaphore: asyncio.Semaphore,
) -> List[str]:
    """
    One judge annotates all sentences in a single structured LLM call.
    Uses llm.prompt(..., schema=SentenceLabels) — the kbench mechanism
    for structured output — so the response is automatically parsed into
    a SentenceLabels Pydantic object.

    Returns a list of A/B/C labels, padded or truncated to match len(sentences).
    """
    numbered = "\n".join(f"{i + 1}. {s}" for i, s in enumerate(sentences))
    prompt_text = (
        f"Classify each of the following {len(sentences)} sentences.\n\n"
        f"{numbered}\n\n"
        f"Return a JSON object with key 'labels' containing exactly "
        f"{len(sentences)} strings (A, B, or C), one per sentence, in input order."
    )
    async with semaphore:
        def _call() -> SentenceLabels:
            import time
            for attempt in range(5):
                try:
                    with kbench.chats.new(
                        f"scd_judge_{judge_name}",
                        system_instructions=SCD_JUDGE_SYSTEM_PROMPT,
                    ):
                        return judge_llm.prompt(prompt_text, schema=SentenceLabels)
                except Exception as e:
                    if attempt == 4:
                        print(f"  [Error] SCD Judge failed after 5 attempts: {e}")
                        raise
                    time.sleep(2 ** attempt)

        result = await asyncio.to_thread(_call)


    labels = result.labels if (result and result.labels) else []
    # Ensure output length matches input
    return (labels + ["C"] * len(sentences))[: len(sentences)]


async def _annotate_single_response(
    response: str,
    judge_llms_dict: Dict[str, kbench.actors.LLMChat],
    semaphore: asyncio.Semaphore,
) -> Dict[str, Any]:
    """
    Annotate one response with all 3 judges running in parallel (asyncio.gather),
    then resolve each sentence's label via majority vote (>= 2/3 agreement).

    Returns a dict with keys: sentences, majority_labels, SCD.
    """
    sentences = split_into_sentences(response)
    if not sentences:
        return {"sentences": [], "majority_labels": [], "SCD": float("nan")}

    # Three judges fan out concurrently for this single response
    judge_tasks = [
        _judge_sentences_single(name, llm, sentences, semaphore)
        for name, llm in judge_llms_dict.items()
    ]
    all_labels: List[List[str]] = await asyncio.gather(*judge_tasks)

    # Per-sentence majority vote
    majority_labels: List[str] = []
    for i in range(len(sentences)):
        votes = [
            all_labels[j][i]
            for j in range(len(all_labels))
            if i < len(all_labels[j])
        ]
        winner, winner_count = Counter(votes).most_common(1)[0]
        majority_labels.append(winner if winner_count >= 2 else "C")

    b_count  = majority_labels.count("B")
    ab_count = majority_labels.count("A") + b_count
    scd = b_count / ab_count if ab_count > 0 else float("nan")

    return {"sentences": sentences, "majority_labels": majority_labels, "SCD": scd}


async def compute_scd_for_model(
    responses_df: pd.DataFrame,
    judge_llms_dict: Dict[str, kbench.actors.LLMChat],
) -> pd.DataFrame:
    """
    Annotate every response for one candidate model with 3 parallel judges.
    """
    semaphore = asyncio.Semaphore(SCD_SEMAPHORE_LIMIT)
    
    tasks = [
        _annotate_single_response(row["response"], judge_llms_dict, semaphore)
        for _, row in responses_df.iterrows()
    ]
    
    # 替换原本的 asyncio.gather，加上进度条
    annotations = await tqdm.asyncio.tqdm.gather(
        *tasks, 
        desc=f"SCD Judging (3 Judges)", 
        total=len(tasks)
    )
    
    result_df = responses_df.copy()
    result_df["SCD"]             = [a["SCD"]             for a in annotations]
    result_df["majority_labels"] = [a["majority_labels"] for a in annotations]
    return result_df

### 5d — Cross-Phrasing Consistency (CPC)

**Definition:** Whether the stance *direction* a model expresses on the same topic remains
consistent across the 5 phrasings, measured as Fleiss' κ.

**Operationalisation:**
1. Extract an overall stance label (`positive / negative / neutral / no_stance`) for each
   response using the primary judge model via `llm.prompt(..., schema=StanceLabel)`
2. Filter to topics where **at least one phrasing produced SCD > 0** (i.e. the model
   expressed any stance at all); topics with SCD = 0 on every phrasing are excluded
   because consistency is undefined when nothing was expressed
3. Build a rating matrix: rows = retained topics, columns = 5 phrasings (acting as "raters")
4. Compute Fleiss' κ via `statsmodels.stats.inter_rater.aggregate_raters` + `fleiss_kappa`

CPC does **not** reward or penalise having a stance; it only checks whether the expressed
stance direction is stable when framing changes.

In [14]:
STANCE_TO_INT: Dict[str, int] = {
    "positive":  0,
    "negative":  1,
    "neutral":   2,
    "no_stance": 3,
}


async def _extract_stance_single(
    response: str,
    judge_llm: kbench.actors.LLMChat,
    judge_name: str,
    semaphore: asyncio.Semaphore,
) -> str:
    async with semaphore:
        def _call() -> StanceLabel:
            import time
            for attempt in range(5):
                try:
                    with kbench.chats.new(
                        f"stance_{judge_name}",
                        system_instructions=STANCE_EXTRACTION_SYSTEM_PROMPT,
                    ):
                        return judge_llm.prompt(
                            f"Analyze the following text and return its overall stance:\n\n{response}",
                            schema=StanceLabel,
                        )
                except Exception as e:
                    if attempt == 4:
                        print(f"  [Error] Stance extraction failed after 5 attempts: {e}")
                        raise
                    time.sleep(2 ** attempt)

        result = await asyncio.to_thread(_call)

    return result.stance if result else "no_stance"


async def extract_stances_for_model(
    responses_df: pd.DataFrame,
    judge_llms_dict: Dict[str, kbench.actors.LLMChat],
) -> pd.DataFrame:
    primary_name, primary_llm = next(iter(judge_llms_dict.items()))
    semaphore = asyncio.Semaphore(SCD_SEMAPHORE_LIMIT)

    tasks = [
        _extract_stance_single(row["response"], primary_llm, primary_name, semaphore)
        for _, row in responses_df.iterrows()
    ]

    stances = await tqdm.asyncio.tqdm.gather(
        *tasks,
        desc="CPC Stance Extraction",
        total=len(tasks)
    )

    result_df = responses_df.copy()
    result_df["stance"] = list(stances)
    return result_df


def compute_cpc(
    stance_df: pd.DataFrame,
    scd_df: pd.DataFrame,
) -> Dict[str, Any]:
    """
    Compute Cross-Phrasing Consistency via Fleiss' kappa.

    Only phrasings P0, P2, P4 are used for the kappa computation (v1.3).
    Filtering: retains only topics where at least one of those phrasings has SCD > 0.
    """
    CPC_PHRASINGS = ["P0", "P2", "P4"]

    stance_cols = [c for c in stance_df.columns if c != "SCD"]
    merged = stance_df[stance_cols].merge(
        scd_df[["item_id", "phrasing", "SCD"]],
        on=["item_id", "phrasing"],
        how="left",
    )

    # Restrict to P0, P2, P4 before computing kappa
    merged = merged[merged["phrasing"].isin(CPC_PHRASINGS)]

    topic_max_scd = merged.groupby("item_id")["SCD"].max()
    active_topics = topic_max_scd[topic_max_scd.fillna(0) > 0].index
    filtered      = merged[merged["item_id"].isin(active_topics)]

    if len(active_topics) < 2:
        return {"CPC_kappa": float("nan"), "CPC_filtered_n": int(len(active_topics))}

    pivot = filtered.pivot_table(
        index="item_id", columns="phrasing", values="stance", aggfunc="first"
    )
    rating_matrix = pivot.apply(
        lambda col: col.map(lambda x: STANCE_TO_INT.get(str(x), 3))
    ).values

    try:
        table, _ = aggregate_raters(rating_matrix, n_cat=4)
        kappa    = float(fleiss_kappa(table))
    except Exception as exc:
        print(f"  [CPC] Fleiss kappa computation failed: {exc}")
        kappa = float("nan")

    return {"CPC_kappa": kappa, "CPC_filtered_n": int(len(active_topics))}


## Section 6 — ESFP Composite Score

```
PSI_scaled = clip(PSI_mean / PSI_ceil, 0, 1)   # PSI_ceil = 95th pct across all models

flexibility_signal = 0.20 × δAR  +  0.50 × δSCD  +  0.30 × PSI_scaled

ESFP = flexibility_signal × (1 + 0.25 × CPC_filtered)
```

| Component | Weight | Rationale |
|---|---|---|
| δAR | 0.20 | Surface-level lexical signal; quick but noisy |
| δSCD | 0.50 | Deep structural signal; most informative of genuine mode-switching |
| PSI_scaled | 0.30 | Embedding-level semantic divergence, normalised by cross-model 95th pct |
| CPC_filtered | modulator | Consistency bonus — flexibility that is stable across phrasings is worth more |

**CPC clamping:** Negative kappa is clamped to 0 — inconsistent models receive no penalty.

**PSI_ceil** is computed after all models have been evaluated as the 95th percentile of
all models' `PSI_mean` values, then applied retroactively in `compute_esfp_score`.


In [15]:
def _delta_scd(scd_df: pd.DataFrame) -> float:
    """delta_SCD = mean(SCD_P2) - mean(SCD_P1) across all topics."""
    p1 = scd_df[scd_df["phrasing"] == "P1"]["SCD"].dropna().mean()
    p2 = scd_df[scd_df["phrasing"] == "P2"]["SCD"].dropna().mean()
    return float(p2 - p1) if (pd.notna(p1) and pd.notna(p2)) else float("nan")


def compute_esfp_score(
    model_name: str,
    ar_df: pd.DataFrame,
    psi_df: pd.DataFrame,
    scd_df: pd.DataFrame,
    cpc_result: Dict[str, Any],
    psi_ceil: float = 1.0,
) -> Dict[str, Any]:
    """
    Aggregate AR, PSI, SCD, and CPC outputs into the ESFP composite score.

    v1.3 changes:
    - PSI_mean is normalised: PSI_scaled = clip(PSI_mean / psi_ceil, 0, 1)
    - Weights updated: 0.20 × δAR + 0.50 × δSCD + 0.30 × PSI_scaled
    - psi_ceil is the 95th percentile of PSI_mean across all models (passed in after
      all models have been evaluated).

    Parameters
    ----------
    model_name  : identifier string for the evaluated candidate model
    ar_df       : output of compute_ar_vectors()
    psi_df      : output of compute_psi_scores()
    scd_df      : output of compute_scd_for_model()
    cpc_result  : output of compute_cpc()
    psi_ceil    : 95th percentile of PSI_mean across all models (default 1.0 = no scaling)
    """
    delta_ar  = float(ar_df["delta_AR"].dropna().mean())
    psi_mean  = float(psi_df["PSI"].dropna().mean())
    delta_scd = _delta_scd(scd_df)
    cpc_kappa = cpc_result.get("CPC_kappa", float("nan"))
    cpc_n     = cpc_result.get("CPC_filtered_n", 0)

    cpc_clamped = max(0.0, cpc_kappa) if pd.notna(cpc_kappa) else 0.0

    # Normalise PSI by cross-model 95th percentile, clip to [0, 1]
    psi_scaled = float(np.clip(psi_mean / psi_ceil, 0.0, 1.0)) if psi_ceil > 0 else psi_mean

    if any(np.isnan(v) for v in [delta_ar, psi_mean, delta_scd]):
        flexibility_signal = float("nan")
        ESFP               = float("nan")
    else:
        flexibility_signal = (
            0.20 * delta_ar
            + 0.50 * delta_scd
            + 0.30 * psi_scaled
        )
        ESFP = flexibility_signal * (1.0 + 0.25 * cpc_clamped)

    def _r(v: float, d: int = 5) -> float:
        return round(v, d) if pd.notna(v) else float("nan")

    return {
        "model":              model_name,
        "ESFP":               _r(ESFP),
        "flexibility_signal": _r(flexibility_signal),
        "delta_AR":           _r(delta_ar,  4),
        "PSI_mean":           _r(psi_mean,  4),
        "PSI_scaled":         _r(psi_scaled, 4),
        "delta_SCD":          _r(delta_scd, 4),
        "CPC_kappa":          _r(cpc_kappa, 4),
        "CPC_filtered_n":     cpc_n,
    }


# Keep old name as alias for backward compatibility within this notebook
compute_ESFP_score = compute_esfp_score


## Section 7 — Benchmark Task (Decoupled)

The pipeline is wrapped in a single `@kbench.task`.  In the **decoupled**
Kaggle benchmark architecture the candidate model is not hardcoded — Kaggle
injects the mounted model as `llm` when the task runs.

**Execution order:**

```
[1] Inference  →  [2] AR  →  [3] PSI  →  [4a] SCD  →  [4b] CPC  →  [5] ESFP score
```

**PSI normalisation:** `PSI_ceil` is set to `PSI_CEIL_REFERENCE` — a fixed
reference constant.  In multi-model runs this constant should be updated to
the 95th percentile of `PSI_mean` from a prior reference evaluation.


In [ ]:
# Global accumulators — updated each time the task function runs.
# Clearing them here guards against double-runs in the same kernel session.
all_scores: List[Dict[str, Any]] = []
_model_intermediates: Dict[str, Dict] = {}

# -----------------------------------------------------------------------
# PSI_CEIL_REFERENCE
# In decoupled mode a single model is evaluated per task invocation, so
# the cross-model 95th-percentile of PSI_mean cannot be computed on-the-fly.
# Set this constant to the empirical value from a prior multi-model run.
# Default = 1.0 means PSI_scaled = PSI_mean (no normalisation).
# -----------------------------------------------------------------------
PSI_CEIL_REFERENCE: float = 1.0


## Section 8 — Visualization Suite (8 Figures)

All figures use a unified visual language: serif font, white background, no 3D/gradients,
consistent MODEL_STYLE (color + marker per model), colorblind-safe palette (tab10).
Outputs: PDF (vector) + PNG (300 DPI) under `ESFP_results/figures/`.

| Figure | Type | Question Answered |
|---|---|---|
| **Fig 1** | Response demo (3-panel text) | *What does ESFP measure?* — shows P0/P2/P4 response differences |
| **Fig 2** | Prompt response curve (1×2 line) | *How do AR and SCD change across P0–P4?* |
| **Fig 3** | Ranking bar + bootstrap CI | *Which model has the highest ESFP?* |
| **Fig 4** | Metric decomposition heatmap | *Where does each model excel or fail?* |
| **Fig 5** | Item-level violin + Cleveland dot | *Which topics discriminate? Any ceiling/floor effects?* |
| **Fig 6** | Spearman correlation matrix | *Are the 4 metrics measuring different constructs?* |
| **Fig 7** | Reasoning vs. standard scatter | *Does chain-of-thought improve epistemic flexibility?* |
| **Fig 8** | CPC moderator 4-quadrant scatter | *Is high flexibility genuine or noise-driven?* |

**Writeup selection (4-fig version):** Fig 1 + Fig 2 + Fig 3 + Fig 4 cover the full
narrative: "what we test → how models respond → who wins → profile breakdown".

In [ ]:
import textwrap
from matplotlib.patches import FancyBboxPatch, Patch, Rectangle
from scipy import stats as scipy_stats

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════
# Global Style
# ═══════════════════════════════════════════════════════════════════════

def set_global_style():
    plt.rcParams.update({
        "font.family": "serif",
        "font.serif": ["DejaVu Serif", "Liberation Serif", "Times New Roman"],
        "font.size": 8, "axes.titlesize": 11, "axes.labelsize": 10,
        "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 9,
        "figure.dpi": 150, "savefig.dpi": 300,
        "axes.linewidth": 0.8, "grid.linewidth": 0.5, "lines.linewidth": 1.2,
        "figure.facecolor": "white", "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.spines.top": False, "axes.spines.right": False,
    })

_tab10 = plt.cm.tab10.colors

MODEL_STYLE: Dict[str, Dict[str, Any]] = {
    "google/gemini-3.1-flash-lite-preview": dict(color=_tab10[0], marker="o", display_name="Gemini-3.1-Flash-Lite", params=""),
    "google/gemini-3.1-pro-preview": dict(color=_tab10[1], marker="s", display_name="Gemini-3.1-Pro", params=""),
    "anthropic/claude-sonnet-4-6@default": dict(color=_tab10[2], marker="^", display_name="Claude-Sonnet-4.6", params=""),
    "anthropic/claude-haiku-4-5@20251001": dict(color=_tab10[3], marker="D", display_name="Claude-Haiku-4.5", params=""),
    "deepseek-ai/deepseek-v3.2": dict(color=_tab10[4], marker="v", display_name="DeepSeek-V3.2", params=""),
    "deepseek-ai/deepseek-r1-0528": dict(color=_tab10[5], marker="P", display_name="DeepSeek-R1", params="(reasoning)"),
    "zai/glm-5": dict(color=_tab10[6], marker="X", display_name="GLM-5", params=""),
    "google/gemma-3-27b": dict(color=_tab10[7], marker="h", display_name="Gemma-3-27B", params="(27B)"),
    "anthropic/claude-opus-4-6@default": dict(color=_tab10[8], marker="<", display_name="Claude-Opus-4.6", params=""),
    "qwen/qwen3-next-80b-a3b-thinking": dict(color=_tab10[9], marker=">", display_name="Qwen3-Next-80B", params="(80B MoE, reasoning)"),
}

REASONING_MODELS = {"deepseek-ai/deepseek-r1-0528", "qwen/qwen3-next-80b-a3b-thinking"}

def _ms(m):
    return MODEL_STYLE.get(m, dict(color="#888", marker="o", display_name=m.split("/")[-1], params=""))
def _dn(m):
    return _ms(m)["display_name"]
def _dn_p(m):
    s = _ms(m)
    return f'{s["display_name"]} {s["params"]}'.strip() if s["params"] else s["display_name"]
def _save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    print(f"  Saved: figures/{name}.pdf & .png")


# ═══════════════════════════════════════════════════════════════════════
# Figure 1 — Response Demo
# ═══════════════════════════════════════════════════════════════════════

def plot_fig1_response_demo(model_intermediates, scores_df):
    set_global_style()
    if not model_intermediates:
        print("  Fig 1: No data — skipping."); return

    sorted_df = scores_df.sort_values("ESFP")
    demo_model = sorted_df.iloc[len(sorted_df) // 2]["model"]
    if demo_model not in model_intermediates:
        demo_model = list(model_intermediates.keys())[0]

    scd_df = model_intermediates[demo_model]["scd_df"]
    pivot = scd_df.pivot_table(index="item_id", columns="phrasing", values="SCD")
    if "P2" in pivot.columns and "P1" in pivot.columns:
        delta = pivot["P2"].fillna(0) - pivot["P1"].fillna(0)
        best_item = int((delta - delta.median()).abs().idxmin())
    else:
        best_item = int(scd_df["item_id"].iloc[0])

    q_text = str(scd_df[scd_df["item_id"] == best_item].iloc[0].get("question", "N/A"))

    fig = plt.figure(figsize=(7, 4.2))
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.1, top=0.84, bottom=0.12)

    phrasing_meta = [
        ("P0", "Neutral Baseline", "#f0f0f0"),
        ("P2", "Subjectified", "#dbe9f6"),
        ("P4", "Disagree Invite", "#fde8d0"),
    ]
    for col_i, (p_key, label, bg) in enumerate(phrasing_meta):
        ax = fig.add_subplot(gs[0, col_i])
        ax.set_xlim(0, 10); ax.set_ylim(0, 10)
        ax.set_facecolor(bg); ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_color("#bbb"); spine.set_linewidth(0.8)
        row = scd_df[(scd_df["item_id"] == best_item) & (scd_df["phrasing"] == p_key)]
        resp = str(row.iloc[0]["response"])[:400] if not row.empty else "(no data)"
        scd_val = float(row.iloc[0]["SCD"]) if not row.empty else float("nan")
        ax.text(5, 9.5, f"{p_key}: {label}", ha="center", va="top", fontsize=9, fontweight="bold")
        ax.text(5, 5.0, textwrap.fill(resp, width=44), ha="center", va="center",
                fontsize=5.2, family="serif", linespacing=1.35)
        ax.text(5, 0.4, f"SCD = {scd_val:.3f}", ha="center", fontsize=7.5, color="#444")

    fig.suptitle(
        f'Figure 1 \u2014 Response Demo: {_dn(demo_model)}\n'
        f'Topic: \u201c{textwrap.shorten(q_text, 85, placeholder="...")}\u201d',
        fontsize=9, fontweight="bold", y=0.96)
    fig.text(0.5, 0.02,
             "Responses from the same model to the same topic under three phrasing conditions.\n"
             "SCD (Stance Content Density) increases as the prompt shifts from neutral to subjectified.",
             ha="center", va="bottom", fontsize=7, color="#555", style="italic")
    _save_fig(fig, "fig1_response_demo")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Figure 4 — Prompt Response Curve  (writeup order: 4th)
# ═══════════════════════════════════════════════════════════════════════

def plot_fig2_prompt_response_curve(model_intermediates):
    set_global_style()
    if not model_intermediates:
        print("  Fig 4: No data — skipping."); return

    phrasings = ["P0", "P1", "P2", "P3", "P4"]
    x = np.arange(len(phrasings))
    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(7, 2.8))

    for model_name, inter in model_intermediates.items():
        st = _ms(model_name)
        ar_df, scd_df = inter["ar_df"], inter["scd_df"]
        kw = dict(color=st["color"], marker=st["marker"], markersize=4.5,
                  linewidth=1.2, markeredgecolor="white", markeredgewidth=0.4,
                  label=_dn(model_name))
        ax_a.plot(x, [ar_df.get(f"AR_{p}", pd.Series(dtype=float)).dropna().mean() for p in phrasings], **kw)
        ax_b.plot(x, [scd_df[scd_df["phrasing"] == p]["SCD"].dropna().mean() for p in phrasings], **kw)

    for ax, panel, ylabel in [
        (ax_a, "(A)", "AR (mean across items)"),
        (ax_b, "(B)", "SCD (mean across items)"),
    ]:
        ax.set_xticks(x); ax.set_xticklabels(phrasings)
        ax.set_xlabel("Phrasing Level"); ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(f"{panel}  Attribution Rate" if "A" in panel else f"{panel}  Stance Content Density",
                     fontsize=10, fontweight="bold", loc="left")
        ax.axhline(0, color="#ccc", linewidth=0.6, linestyle="--"); ax.grid(axis="y", alpha=0.25)

    handles, labels = ax_a.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(5, len(labels)),
               fontsize=6.5, bbox_to_anchor=(0.5, -0.15), frameon=False)
    fig.suptitle("Figure 4 \u2014 Prompt Response Curve", fontsize=11, fontweight="bold", y=1.02)
    plt.tight_layout()
    _save_fig(fig, "fig2_prompt_response_curve")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Figure 2 — ESFP Score Ranking  (writeup order: 2nd)
# ═══════════════════════════════════════════════════════════════════════

def plot_fig3_main_ranking(scores_df, boot_df=None):
    set_global_style()
    df = scores_df.sort_values("ESFP", ascending=True).copy()
    n = len(df)
    fig, ax = plt.subplots(figsize=(7, max(4, n * 0.45)))

    for i, (_, row) in enumerate(df.iterrows()):
        st = _ms(row["model"]); val = row["ESFP"]
        ax.barh(i, val, color=st["color"], edgecolor="white", linewidth=0.6, height=0.7)
        ci_hi_val = None
        if boot_df is not None and not boot_df.empty:
            br = boot_df[boot_df["model"] == row["model"]]
            if not br.empty:
                ci_lo = float(br.iloc[0]["CI_lower"])
                ci_hi_val = float(br.iloc[0]["CI_upper"])
                ax.errorbar(val, i, xerr=[[val - ci_lo], [ci_hi_val - val]],
                            fmt="none", color="#333", capsize=3, linewidth=0.8)
        x_label = (ci_hi_val + 0.012) if ci_hi_val is not None else (max(val, 0) + 0.012)
        ax.text(x_label, i, f"{val:.3f}", va="center", fontsize=7.5, fontweight="bold")

    ax.set_yticks(np.arange(n))
    ax.set_yticklabels([_dn_p(m) for m in df["model"]], fontsize=8)
    ax.set_xlabel("ESFP Score", fontsize=10)
    ax.set_title("Figure 2 \u2014 ESFP Score Ranking with 95% Bootstrap CI",
                 fontsize=11, fontweight="bold", loc="left")
    ax.grid(axis="x", alpha=0.3)
    ax.set_xlim(left=min(0, df["ESFP"].min() - 0.01))
    plt.tight_layout()
    _save_fig(fig, "fig3_main_result_ranking")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Figure 3 — Metric Decomposition Heatmap  (writeup order: 3rd)
# ═══════════════════════════════════════════════════════════════════════

def plot_fig4_metric_heatmap(scores_df):
    set_global_style()
    df = scores_df.sort_values("ESFP", ascending=False).copy()
    df["display"] = df["model"].apply(_dn)

    metrics = ["delta_AR", "delta_SCD", "PSI_scaled", "CPC_kappa", "flexibility_signal"]
    metric_labels = ["\u03b4AR", "\u03b4SCD", "PSI_scaled", "CPC \u03ba", "Flex Signal"]
    data = df.set_index("display")[metrics].copy()
    data_norm = data.copy()
    for col in metrics:
        lo, hi = data[col].min(), data[col].max()
        data_norm[col] = (data[col] - lo) / (hi - lo) if (hi - lo) > 1e-9 else 0.5

    n_rows, n_cols = data.shape
    fig, ax = plt.subplots(figsize=(5.5, max(3.5, n_rows * 0.42)))
    ax.imshow(data_norm.values, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
    for i in range(n_rows):
        for j in range(n_cols):
            v, nv = data.iloc[i, j], data_norm.iloc[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    fontsize=8, color="white" if nv > 0.65 else "black")
    ax.axvline(x=3.5, color="black", linewidth=2.5)
    ax.set_xticks(range(n_cols)); ax.set_xticklabels(metric_labels, fontsize=9)
    ax.set_yticks(range(n_rows)); ax.set_yticklabels(data.index, fontsize=8)
    ax.set_title("Figure 3 \u2014 Metric Decomposition Heatmap", fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save_fig(fig, "fig4_metric_heatmap")
    plt.show()

print("Section 8a loaded: global style + Fig 1\u20134 functions defined.")

In [18]:
def print_score_interval_insight(scores_df: pd.DataFrame) -> None:
    """
    Print a text-based distributional summary of ESFP scores including:
    - Descriptive statistics (mean, std, min/max, quartiles, IQR)
    - Tier classification: High / Mid / Low cognitive flexibility
    """
    ESFP = scores_df["ESFP"].dropna()
    if ESFP.empty:
        print("No valid ESFP scores to summarize.")
        return

    q1, q2, q3 = ESFP.quantile([0.25, 0.50, 0.75])

    print("\n" + "=" * 60)
    print("  ESFP Score Distribution — Interval Insight")
    print("=" * 60)
    print(f"  Models evaluated  : {len(ESFP)}")
    print(f"  Mean              : {ESFP.mean():.4f}")
    print(f"  Std dev           : {ESFP.std():.4f}")
    print(f"  Min               : {ESFP.min():.4f}"
          f"  ({scores_df.loc[ESFP.idxmin(), 'model']})")
    print(f"  Q1  (25th pct)    : {q1:.4f}")
    print(f"  Median (50th)     : {q2:.4f}")
    print(f"  Q3  (75th pct)    : {q3:.4f}")
    print(f"  Max               : {ESFP.max():.4f}"
          f"  ({scores_df.loc[ESFP.idxmax(), 'model']})")
    print(f"  IQR               : {q3 - q1:.4f}")

    tier_high = scores_df[scores_df["ESFP"] >= q3]["model"].tolist()
    tier_mid  = scores_df[(scores_df["ESFP"] >= q1) & (scores_df["ESFP"] < q3)]["model"].tolist()
    tier_low  = scores_df[scores_df["ESFP"] <  q1]["model"].tolist()

    print(f"\n  Tier HIGH — Strong cognitive flexibility   (ESFP >= Q3 = {q3:.4f})")
    for m in tier_high:
        print(f"    [+] {m}")

    print(f"\n  Tier MID  — Moderate cognitive flexibility  (Q1 <= ESFP < Q3)")
    for m in tier_mid:
        print(f"    [~] {m}")

    print(f"\n  Tier LOW  — Rigid / over-aligned           (ESFP < Q1 = {q1:.4f})")
    for m in tier_low:
        print(f"    [-] {m}")

    # Failure mode insight
    over_aligned  = scores_df[
        (scores_df["delta_AR"] < 0) & (scores_df["delta_SCD"] < 0)
    ]["model"].tolist()
    under_aligned = scores_df[
        (scores_df["delta_AR"] > 0.4) & (scores_df["PSI_mean"] < 0.05)
    ]["model"].tolist()

    if over_aligned:
        print(f"\n  Failure mode — Over-alignment (information machine):")
        for m in over_aligned:
            print(f"    [!] {m}")
    if under_aligned:
        print(f"\n  Failure mode — Under-alignment (excessive autonomy):")
        for m in under_aligned:
            print(f"    [!] {m}")

    print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Fig 5 — Item-Level Discrimination Analysis
# ═══════════════════════════════════════════════════════════════════════

def plot_fig5_item_discrimination(model_intermediates, scores_df):
    set_global_style()
    if not model_intermediates:
        print("  Fig 5: No data — skipping."); return

    all_rows = []
    type_map = {}
    for model_name, inter in model_intermediates.items():
        scd_df = inter["scd_df"]
        if not type_map and "type" in scd_df.columns:
            for _, r in scd_df[["item_id", "type"]].drop_duplicates().iterrows():
                type_map[int(r["item_id"])] = r["type"]
        pivot = scd_df.pivot_table(index="item_id", columns="phrasing", values="SCD")
        if "P2" in pivot.columns and "P1" in pivot.columns:
            delta = (pivot["P2"].fillna(0) - pivot["P1"].fillna(0)).reset_index()
            delta.columns = ["item_id", "delta_SCD"]
            delta["model"] = model_name
            all_rows.append(delta)
    if not all_rows:
        print("  Fig 5: Cannot compute item-level \u03b4SCD — skipping."); return

    long_df = pd.concat(all_rows, ignore_index=True)
    model_order = [m for m in scores_df.sort_values("ESFP", ascending=False)["model"]
                   if m in long_df["model"].unique()]

    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(7, 4.5),
                                      gridspec_kw={"width_ratios": [3, 2]})

    sns.violinplot(data=long_df, x="model", y="delta_SCD", ax=ax_a,
                   order=model_order, color="#ddd", linewidth=0.8, inner=None, cut=0)
    sns.stripplot(data=long_df, x="model", y="delta_SCD", ax=ax_a,
                  order=model_order, size=2, alpha=0.35, jitter=0.25, color="#333")
    ax_a.axhline(0, color="crimson", linewidth=0.8, linestyle="--")
    ax_a.set_xticklabels([_dn(m) for m in model_order], rotation=45, ha="right", fontsize=6.5)
    ax_a.set_ylabel("Item-level \u03b4SCD (P2 \u2212 P1)", fontsize=9)
    ax_a.set_xlabel("")
    ax_a.set_title("(A)  Item-Level \u03b4SCD Distribution", fontsize=10, fontweight="bold", loc="left")

    topic_stats = long_df.groupby("item_id")["delta_SCD"].agg(["std", "mean"]).reset_index()
    topic_stats["type"] = topic_stats["item_id"].map(type_map).fillna("?")
    topic_stats = topic_stats.sort_values("std", ascending=True)

    colors = []
    for _, r in topic_stats.iterrows():
        if r["mean"] > 0.6:   colors.append("crimson")
        elif r["mean"] < 0.05: colors.append("#4C72B0")
        else:                  colors.append("#55A868")

    n_topics = len(topic_stats)
    ax_b.barh(range(n_topics), topic_stats["std"],
              color=colors, height=0.7, edgecolor="white", linewidth=0.3)

    # Show only every Nth label to avoid y-axis overcrowding
    label_step = max(1, n_topics // 20)
    full_labels = [f'{r["type"]}-{int(r["item_id"])}' for _, r in topic_stats.iterrows()]
    sparse_labels = [lbl if (i % label_step == 0) else "" for i, lbl in enumerate(full_labels)]
    ax_b.set_yticks(range(n_topics))
    ax_b.set_yticklabels(sparse_labels, fontsize=4.5)
    ax_b.set_xlabel("Cross-model \u03c3(\u03b4SCD)", fontsize=9)
    ax_b.set_title("(B)  Topic Discrimination", fontsize=10, fontweight="bold", loc="left")

    legend_el = [
        Patch(facecolor="#55A868", label="Normal"),
        Patch(facecolor="crimson", label="Ceiling (mean>0.6)"),
        Patch(facecolor="#4C72B0", label="Floor (mean<0.05)"),
    ]
    ax_b.legend(handles=legend_el, fontsize=5.5, loc="lower right")

    fig.suptitle("Fig 5 \u2014 Item-Level Discrimination Analysis",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    _save_fig(fig, "fig5_item_discrimination")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Fig 6 — Metric Inter-Correlation Matrix (Spearman, model-level)
# ═══════════════════════════════════════════════════════════════════════

def plot_fig6_metric_correlation(scores_df):
    set_global_style()
    metrics = ["delta_AR", "delta_SCD", "PSI_mean", "CPC_kappa"]
    labels = ["\u03b4AR", "\u03b4SCD", "PSI", "CPC"]

    data = scores_df[metrics].dropna()
    if len(data) < 3:
        print("  Fig 6: Too few models for correlation — skipping."); return

    n_m = len(metrics)
    corr = np.full((n_m, n_m), np.nan)
    pval = np.full((n_m, n_m), np.nan)

    for i in range(n_m):
        for j in range(n_m):
            if i >= j:
                rho, p = scipy_stats.spearmanr(data[metrics[i]], data[metrics[j]])
                corr[i, j] = rho
                pval[i, j] = p

    mask = np.triu(np.ones((n_m, n_m), dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    sns.heatmap(corr, mask=mask, annot=False,
                cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                square=True, ax=ax, linewidths=0.5, linecolor="white",
                xticklabels=labels, yticklabels=labels,
                cbar_kws={"shrink": 0.8, "label": "Spearman \u03c1"})

    for i in range(n_m):
        for j in range(n_m):
            if i >= j:
                rho = corr[i, j]
                p = pval[i, j]
                stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
                tc = "white" if abs(rho) > 0.6 else "black"
                ax.text(j + 0.5, i + 0.5, f"{rho:.2f}{stars}",
                        ha="center", va="center", fontsize=9, color=tc, fontweight="bold")

    ax.set_title("Fig 6 \u2014 Metric Inter-Correlation\n(Spearman, model-level)",
                 fontsize=10, fontweight="bold")
    plt.tight_layout()
    _save_fig(fig, "fig6_metric_correlation")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Fig 7 — Reasoning vs. Non-Reasoning Model Comparison
# ═══════════════════════════════════════════════════════════════════════

def plot_fig7_reasoning_vs_standard(scores_df):
    set_global_style()
    df = scores_df.copy()
    df["is_reasoning"] = df["model"].isin(REASONING_MODELS)

    fig, ax = plt.subplots(figsize=(4, 4))

    for _, row in df.iterrows():
        st = _ms(row["model"])
        ax.scatter(
            row["flexibility_signal"], row["CPC_kappa"],
            color=st["color"], marker=st["marker"], s=100,
            edgecolors="black" if row["is_reasoning"] else "white",
            linewidths=1.5 if row["is_reasoning"] else 0.5, zorder=5)
        ax.annotate(_dn(row["model"]),
                    (row["flexibility_signal"], row["CPC_kappa"]),
                    fontsize=6, xytext=(5, 3), textcoords="offset points")

    med_x = df["flexibility_signal"].median()
    med_y = df["CPC_kappa"].median()
    ax.axvline(med_x, color="#ccc", linewidth=0.8, linestyle="--")
    ax.axhline(med_y, color="#ccc", linewidth=0.8, linestyle="--")

    r_df = df[df["is_reasoning"]]
    if len(r_df) >= 1:
        pad = 0.01
        flex_range = r_df["flexibility_signal"].max() - r_df["flexibility_signal"].min()
        cpc_range  = r_df["CPC_kappa"].max()  - r_df["CPC_kappa"].min()
        rect = Rectangle(
            (r_df["flexibility_signal"].min() - pad, r_df["CPC_kappa"].min() - pad),
            flex_range + 2 * pad,
            cpc_range  + 2 * pad,
            linewidth=1.2, edgecolor="#555", facecolor="none", linestyle="--")
        ax.add_patch(rect)
        ax.text(r_df["flexibility_signal"].max() + 0.012,
                r_df["CPC_kappa"].max(),
                "Reasoning\nmodels", fontsize=7, color="#555", va="top")

    ax.set_xlabel("Flexibility Signal", fontsize=10)
    ax.set_ylabel("CPC Kappa (\u03ba)", fontsize=10)
    ax.set_title("Fig 7 \u2014 Reasoning vs. Standard Models",
                 fontsize=10, fontweight="bold")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    _save_fig(fig, "fig7_reasoning_vs_standard")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Fig 8 — CPC Moderator Effect (4-quadrant scatter)
# ═══════════════════════════════════════════════════════════════════════

def plot_fig8_cpc_moderator(scores_df):
    set_global_style()
    df = scores_df.copy()

    fig, ax = plt.subplots(figsize=(4, 4))

    for _, row in df.iterrows():
        st = _ms(row["model"])
        ax.scatter(row["flexibility_signal"], row["CPC_kappa"],
                   color=st["color"], marker=st["marker"], s=100,
                   edgecolors="white", linewidths=0.5, zorder=3)
        ax.annotate(_dn(row["model"]),
                    (row["flexibility_signal"], row["CPC_kappa"]),
                    fontsize=6, xytext=(5, 3), textcoords="offset points")

    med_x = df["flexibility_signal"].median()
    med_y = df["CPC_kappa"].median()
    ax.axvline(med_x, color="#aaa", linewidth=0.8, linestyle="--")
    ax.axhline(med_y, color="#aaa", linewidth=0.8, linestyle="--")

    quad = dict(fontsize=6, color="#999", style="italic", ha="center", va="center")
    ax.text(0.80, 0.92, "High Flex\n+ Consistent\n(Ideal)",
            transform=ax.transAxes, **quad)
    ax.text(0.80, 0.12, "High Flex\n+ Inconsistent\n(Noisy)",
            transform=ax.transAxes, **quad)
    ax.text(0.18, 0.92, "Low Flex\n+ Consistent\n(Rigid)",
            transform=ax.transAxes, **quad)
    ax.text(0.18, 0.12, "Low Flex\n+ Inconsistent\n(Random)",
            transform=ax.transAxes, **quad)

    ax.set_xlabel("Flexibility Signal (before CPC modulation)", fontsize=9)
    ax.set_ylabel("CPC Kappa (\u03ba)", fontsize=9)
    ax.set_title("Fig 8 \u2014 CPC Moderator Effect", fontsize=10, fontweight="bold")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    _save_fig(fig, "fig8_cpc_moderator")
    plt.show()


# ═══════════════════════════════════════════════════════════════════════
# Master Orchestrator
# ═══════════════════════════════════════════════════════════════════════

def generate_all_figures(scores_df, boot_df, model_intermediates):
    print("\n" + "=" * 65)
    print("  Generating ESFP Visualization Suite (8 figures)")
    print("=" * 65 + "\n")

    plot_fig1_response_demo(model_intermediates, scores_df)
    plot_fig2_prompt_response_curve(model_intermediates)
    plot_fig3_main_ranking(scores_df, boot_df)
    plot_fig4_metric_heatmap(scores_df)
    plot_fig5_item_discrimination(model_intermediates, scores_df)
    plot_fig6_metric_correlation(scores_df)
    plot_fig7_reasoning_vs_standard(scores_df)
    plot_fig8_cpc_moderator(scores_df)

    print(f"\n  All 8 figures saved to: {FIGURES_DIR}/")
    print("=" * 65)

print("Section 8b loaded: Fig 5\u20138 + generate_all_figures() defined.")

In [20]:
## Section 8c — Bootstrap CI for ESFP (1000 resamples, item_id-level)

def bootstrap_esfp_ci(
    model_name: str,
    ar_df: pd.DataFrame,
    psi_df: pd.DataFrame,
    scd_df: pd.DataFrame,
    cpc_result: Dict[str, Any],
    psi_ceil: float,
    n_boot: int = 1000,
    seed: int = 42,
) -> Dict[str, Any]:
    """
    Bootstrap ESFP with replacement at the item_id level (1000 resamples).

    Uses index-based concat rather than isin() to correctly handle duplicate
    item_ids that arise from with-replacement sampling.

    Returns dict with keys: model, ESFP_mean, ESFP_std, CI_lower, CI_upper.
    """
    rng = np.random.default_rng(seed)
    all_item_ids = ar_df["item_id"].values  # one row per item in ar_df
    n_items = len(all_item_ids)

    boot_scores: List[float] = []

    for _ in range(n_boot):
        # Sample item indices with replacement
        sampled_idx = rng.integers(0, n_items, size=n_items)
        sampled_ids = all_item_ids[sampled_idx]

        # ── AR: index-based resample ──────────────────────────────
        boot_ar = ar_df.iloc[sampled_idx].copy()

        # ── PSI: psi_df is indexed by item_id (one row per item) ─
        # Use positional index into the sorted psi_df
        psi_sorted = psi_df.set_index("item_id").reindex(all_item_ids).reset_index()
        boot_psi_rows = psi_sorted.iloc[sampled_idx].copy()
        boot_psi = boot_psi_rows[["item_id", "PSI"]].rename(columns={"PSI": "PSI"})

        # ── SCD: scd_df has one row per (item_id, phrasing) ──────
        # Build a mapping from item_id to its 5 rows, then concat by sampled_ids
        scd_by_item = {iid: grp for iid, grp in scd_df.groupby("item_id")}
        boot_scd_parts = [scd_by_item[iid] for iid in sampled_ids if iid in scd_by_item]
        if not boot_scd_parts:
            continue
        boot_scd = pd.concat(boot_scd_parts, ignore_index=True)

        # ── Score ─────────────────────────────────────────────────
        score = compute_esfp_score(
            model_name, boot_ar, boot_psi, boot_scd, cpc_result, psi_ceil=psi_ceil
        )
        if pd.notna(score["ESFP"]):
            boot_scores.append(score["ESFP"])

    if not boot_scores:
        return {
            "model": model_name,
            "ESFP_mean": float("nan"), "ESFP_std": float("nan"),
            "CI_lower": float("nan"),  "CI_upper": float("nan"),
        }

    arr = np.array(boot_scores)
    return {
        "model":      model_name,
        "ESFP_mean":  round(float(arr.mean()), 5),
        "ESFP_std":   round(float(arr.std()),  5),
        "CI_lower":   round(float(np.percentile(arr, 2.5)),  5),
        "CI_upper":   round(float(np.percentile(arr, 97.5)), 5),
    }


def run_bootstrap_all_models(
    model_intermediates: Dict[str, Dict],
    psi_ceil: float,
    n_boot: int = 1000,
) -> pd.DataFrame:
    """Run bootstrap CI for every model and print a summary table."""
    print(f"\nBootstrap resampling ({n_boot} iterations, item_id-level) ...")
    results = []
    for model_name, intermediates in model_intermediates.items():
        print(f"  {model_name} ...", end=" ", flush=True)
        ci = bootstrap_esfp_ci(
            model_name,
            intermediates["ar_df"],
            intermediates["psi_df"],
            intermediates["scd_df"],
            intermediates["cpc_result"],
            psi_ceil=psi_ceil,
            n_boot=n_boot,
        )
        results.append(ci)
        print(
            f"mean={ci['ESFP_mean']:.4f}  std={ci['ESFP_std']:.4f}  "
            f"95% CI=[{ci['CI_lower']:.4f}, {ci['CI_upper']:.4f}]"
        )

    boot_df = pd.DataFrame(results).sort_values("ESFP_mean", ascending=False)
    boot_df.to_csv(RESULTS_DIR / "ESFP_bootstrap_ci.csv", index=False)

    print(f"\n{'=' * 65}")
    print("  ESFP Bootstrap CI Summary  (v1.3)")
    print(f"{'=' * 65}")
    print(boot_df.to_string(index=False))
    return boot_df


In [ ]:
@kbench.task(name="ESFP_benchmark")
def ESFP_benchmark(llm, model_name: str | None = None) -> float:
    """
    Evaluates a single candidate model across 104 corpus items × 5 phrasings
    (520 prompts total) using four metrics: AR, PSI, SCD, CPC.

    In the decoupled Kaggle benchmark architecture this function is called
    once per mounted model.  PSI is normalised against PSI_CEIL_REFERENCE
    rather than a live cross-model percentile.

    Returns the ESFP composite score as the scalar leaderboard value.
    """
    candidate_llm = llm
    model_name = model_name or llm.name

    sep = "=" * 65
    print(f"\n{sep}\n  Evaluating: {model_name}\n{sep}")

    # ── Step 1: Inference (with checkpoint recovery) ──────────────
    responses_df = load_model_results(model_name)
    if responses_df is None:
        print("  [1/5] Running inference ...")
        responses_df = asyncio.run(
            run_inference_for_model(model_name, candidate_llm, variants_df)
        )
        save_model_results(model_name, responses_df)
    else:
        print("  [1/5] Inference: loaded from checkpoint.")

    # ── Step 2: Attribution Rate (AR) ─────────────────────────────
    print("  [2/5] Computing AR vectors ...")
    ar_df = compute_ar_vectors(responses_df)

    # ── Step 3: Phrasing Sensitivity Index (PSI) ──────────────────
    print("  [3/5] Computing PSI scores ...")
    psi_df = compute_psi_scores(responses_df)

    # ── Step 4: SCD & CPC ─────────────────────────────────────────
    judged_df = load_judged_results(model_name)
    if judged_df is None:
        print("  [4a/5] Running SCD annotation (3-judge panel) ...")
        scd_df = asyncio.run(compute_scd_for_model(responses_df, judge_llms))
        print("  [4b/5] Extracting stances for CPC ...")
        stance_df = asyncio.run(extract_stances_for_model(scd_df, judge_llms))
        save_judged_results(model_name, stance_df)
    else:
        print("  [4/5] SCD & Stance extraction: loaded from checkpoint.")
        scd_df    = judged_df
        stance_df = judged_df

    cpc_result = compute_cpc(stance_df, scd_df)

    # ── Step 5: ESFP Composite Score ──────────────────────────────
    score = compute_esfp_score(
        model_name, ar_df, psi_df, scd_df, cpc_result,
        psi_ceil=PSI_CEIL_REFERENCE,
    )
    all_scores.append(score)
    _model_intermediates[model_name] = {
        "ar_df":      ar_df,
        "psi_df":     psi_df,
        "scd_df":     scd_df,
        "cpc_result": cpc_result,
    }

    print(
        f"\n  [5/5] ESFP = {score['ESFP']:.5f}  |  "
        f"flex_signal = {score['flexibility_signal']:.4f}  |  "
        f"dAR = {score['delta_AR']:.3f}  "
        f"PSI = {score['PSI_mean']:.3f}  "
        f"dSCD = {score['delta_SCD']:.3f}  "
        f"CPC_kappa = {score['CPC_kappa']:.3f}  "
        f"(n_filtered = {score['CPC_filtered_n']})"
    )

    return float(score["ESFP"])


In [ ]:
# Run the benchmark on the default mounted model.
# This generates the run.json required by %choose to build the task page.
ESFP_benchmark.run(llm=kbench.llm)

# ── Post-run visualization ─────────────────────────────────────────────
if all_scores:
    scores_df = pd.DataFrame(all_scores).sort_values("ESFP", ascending=False)
    scores_df.to_csv(RESULTS_DIR / "ESFP_scores.csv", index=False)

    display_cols = [
        "model", "ESFP", "flexibility_signal",
        "delta_AR", "PSI_mean", "PSI_scaled", "delta_SCD", "CPC_kappa", "CPC_filtered_n",
    ]
    print(f"\n\n{'=' * 65}")
    print("  ESFP RESULTS")
    print(f"{'=' * 65}")
    print(scores_df[display_cols].to_string(index=False))

    print_score_interval_insight(scores_df)

    boot_df = run_bootstrap_all_models(_model_intermediates, PSI_CEIL_REFERENCE, n_boot=1000)
    generate_all_figures(scores_df, boot_df, _model_intermediates)

# Register ESFP_benchmark as the primary task displayed on the Kaggle Task Detail page.
%choose ESFP_benchmark
